In [1]:
import pandas as pd

# 1. Load cleaned data (file must be in the same folder as this notebook)
df = pd.read_csv("output/dataset_mansion_cleaned.csv", parse_dates=['MTime'])
df = df.sort_values('MTime').reset_index(drop=True)

df.head()


,MTime,Unnamed: 0,Consumption,Latitude,Longitude,Place,AirTemperature(degC),HighestTemperature(degC),LowestTemperature(degC),RelativeHumidity(%),...,weather_missing,hour,dow,month,is_weekend,heating_season,hour_sin,hour_cos,doy_sin,doy_cos
0,2015-12-31 22:00:00,0,2.0,62.39758,25.67087,Jyväskylä lentoasema,-4.3,-4.2,-4.5,71.0,...,0,22,3,12,0,1,-0.500000,0.866025,-0.004301,0.999991
1,2015-12-31 23:00:00,1,2.0,62.39758,25.67087,Jyväskylä lentoasema,-4.6,-4.4,-4.6,73.0,...,0,23,3,12,0,1,-0.258819,0.965926,-0.004301,0.999991
2,2016-01-01 00:00:00,2,2.0,62.39758,25.67087,Jyväskylä lentoasema,-4.5,-4.4,-4.5,71.0,...,0,0,4,1,0,1,0.000000,1.000000,0.017202,0.999852
3,2016-01-01 01:00:00,3,2.0,62.39758,25.67087,Jyväskylä lentoasema,-4.5,-4.5,-4.6,71.0,...,0,1,4,1,0,1,0.258819,0.965926,0.017202,0.999852
4,2016-01-01 02:00:00,4,2.0,62.39758,25.67087,Jyväskylä lentoasema,-4.5,-4.5,-4.6,72.0,...,0,2,4,1,0,1,0.500000,0.866025,0.017202,0.999852


In [2]:
# Drop constant / non-useful columns for modeling
for col in ['Latitude', 'Longitude', 'Place']:
    if col in df.columns:
        df = df.drop(columns=[col])

# ===== Weather-based features =====
temp = df['AirTemperature(degC)']

# Heating / cooling degree-type features
base_temp = 18
df['temp_below_18'] = (base_temp - temp).clip(lower=0)
df['temp_above_18'] = (temp - base_temp).clip(lower=0)

# Wind-chill / feels-like temperature
v = df['WindSpeed(m/s)']
df['wind_chill'] = (
    13.12
    + 0.6215 * temp
    - 11.37 * (v ** 0.16)
    + 0.3965 * temp * (v ** 0.16)
)

# Precipitation indicator
df['is_precip'] = (df['PrecipitationAmount(mm)'] > 0).astype(int)

# PresentWeather missing flag + fill
df['weather_rank_missing'] = df['PresentWeather(rank)'].isna().astype(int)
df['PresentWeather(rank)'] = df['PresentWeather(rank)'].fillna(-1)

# ===== Extra time-of-day features =====
df['is_night'] = df['hour'].isin([0, 1, 2, 3, 4, 5, 23]).astype(int)
df['is_work_hours'] = df['hour'].between(8, 17).astype(int)
df['is_evening'] = df['hour'].between(18, 22).astype(int)

# ===== Lag features for consumption =====
for lag in [1, 2, 3, 24, 168]:  # 1h, 2h, 3h, 1 day, 1 week
    df[f'cons_lag_{lag}'] = df['Consumption'].shift(lag)

# ===== Rolling statistics of consumption =====
for window in [3, 6, 24, 168]:
    df[f'cons_roll_mean_{window}'] = (
        df['Consumption'].shift(1).rolling(window=window).mean()
    )
    df[f'cons_roll_std_{window}'] = (
        df['Consumption'].shift(1).rolling(window=window).std()
    )

# ===== Optional: lagged temperature =====
for lag in [1, 3]:
    df[f'temp_lag_{lag}'] = df['AirTemperature(degC)'].shift(lag)

# ===== Target: next-hour consumption =====
df['y_next_1h'] = df['Consumption'].shift(-1)

df.head()


,MTime,Unnamed: 0,Consumption,AirTemperature(degC),HighestTemperature(degC),LowestTemperature(degC),RelativeHumidity(%),WindSpeed(m/s),MaximumWindSpeed(m/s),MinimumWindSpeed(m/s),...,cons_roll_std_3,cons_roll_mean_6,cons_roll_std_6,cons_roll_mean_24,cons_roll_std_24,cons_roll_mean_168,cons_roll_std_168,temp_lag_1,temp_lag_3,y_next_1h
0,2015-12-31 22:00:00,0,2.0,-4.3,-4.2,-4.5,71.0,3.8,4.9,2.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
1,2015-12-31 23:00:00,1,2.0,-4.6,-4.4,-4.6,73.0,3.1,3.7,2.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.3,NaN,2.0
2,2016-01-01 00:00:00,2,2.0,-4.5,-4.4,-4.5,71.0,3.8,4.3,3.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.6,NaN,2.0
3,2016-01-01 01:00:00,3,2.0,-4.5,-4.5,-4.6,71.0,4.0,4.5,3.3,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,-4.5,-4.3,2.0
4,2016-01-01 02:00:00,4,2.0,-4.5,-4.5,-4.6,72.0,4.1,4.7,3.5,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,-4.5,-4.6,2.0


In [3]:
df.to_csv("output/dataset_mansion_features.csv", index=False)
